# Annotation & training-set builder

Replaces the hand-edited dict-literal workflow. Key differences:

- **Nothing is edited in code cells.** Corrections are made in a widget and written to
  `annotations/<company>.json`. Source OCR output is never mutated.
- **Every page is keyed by `<company>/<page_file>`**, never by list position.
- **Every save is validated** (JSON parses, row lengths match headers, no corrupted labels)
  and written atomically, so a bad edit can't silently vanish.
- **Negatives are included.** Pages the base model said had no table enter the queue too,
  so detection recall is trainable and measurable.
- **Splits are by company**, never by page, so no filing appears on both sides.

Expected layout:

```
ROOT/
  company_filing_pages/<company_number>/page_001.png ...
  raw_ocr/<company_number>.json
  annotations/<company_number>.json      <- created here
  exports/                               <- created here
```

In [ ]:
# --- cell 1: setup -------------------------------------------------------
# In Colab, uncomment:
# from google.colab import drive; drive.mount('/content/drive')
# !pip install -q ipywidgets pandas pillow

import json, os, random, re, shutil, tempfile, hashlib
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

# ---- EDIT THESE ---------------------------------------------------------
ROOT       = Path.cwd()
PAGES_DIR  = ROOT / 'company_filing_pages'
OCR_DIR    = ROOT / 'raw_ocr'
ANNOT_DIR  = ROOT / 'annotations'
EXPORT_DIR = ROOT / 'exports'
SEED       = 20250816
# -------------------------------------------------------------------------

ANNOT_DIR.mkdir(exist_ok=True)
EXPORT_DIR.mkdir(exist_ok=True)
random.seed(SEED)

for p in (PAGES_DIR, OCR_DIR):
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

OK   d:\University\Masters\Project\team_21_lloyds\Data Preprocess\company_filing_pages
MISSING  d:\University\Masters\Project\team_21_lloyds\Data Preprocess\company_ocr_results


In [3]:
# --- cell 3: index every page -------------------------------------------
# Page keys are always "<company>/<filename>". OCR result keys in the raw
# files are absolute Colab paths, so they get normalised to the basename.

PAGE_RE = re.compile(r'page_(\d+)\.(png|jpg|jpeg)$', re.I)

# returns last component of a path
# Parameters: 
#               raw_key: str - a string representing a file path
# Returns: 
#               str - the last component of the path (the file name)
def norm_key(raw_key: str) -> str:
    """'/content/company_filing/12273666/page_005.png' -> 'page_005.png'"""
    return Path(str(raw_key)).name

# returns a dict of OCR results for a given company, normalising the keys to just the page file name
# Parameters: 
#               company: str - the target company number 
# Returns: 
#               dict - a dictionary containing the OCR results for the given company, with keys normalised to just the page file name
def load_ocr(company: str) -> dict:
    f = OCR_DIR / f'{company}.json'
    if not f.exists():
        return {}

    try:
        raw = json.loads(f.read_text(encoding='utf-8'))
    except json.JSONDecodeError as e:
        print(f'  ! {company}.json failed to parse: {e}')
        return {}

    if not isinstance(raw, dict):
        return {}
    
    out = {}
    for k, v in raw.items():
        # Pipeline wrote {page: [table, ...]} for financial_tables, but
        # {page: {"table_found":..,"tables":[..]}} elsewhere. Accept both.
        if isinstance(v, list):
            v = v[0]
        if isinstance(v, dict):
            out[norm_key(k)] = v
    return out


def build_index() -> pd.DataFrame:
    rows = []
    companies = sorted(d.name for d in PAGES_DIR.iterdir() if d.is_dir())

    # loop through each company's ocr result
    for company in companies:
        ocr = load_ocr(company)
        imgs = sorted(p for p in (PAGES_DIR / company).iterdir() if PAGE_RE.search(p.name))


        for img in imgs:
            rec = ocr.get(img.name)
            tf = bool(rec.get('table_found')) if isinstance(rec, dict) else None
            tables = rec.get('tables') or [] if isinstance(rec, dict) else []
            rows.append({
                'key':          f'{company}/{img.name}',
                'company':      company,
                'page_file':    img.name,
                'page_no':      int(PAGE_RE.search(img.name).group(1)),
                'image_path':   str(img),
                'has_ocr':      rec is not None,
                'ocr_found':    tf,
                'ocr_n_tables': len(tables) if isinstance(tables, list) else 0,
            })
            
    return pd.DataFrame(rows).sort_values(['company', 'page_no']).reset_index(drop=True)

index_df = build_index()
print(f"{len(index_df)} pages across {index_df['company'].nunique()} companies")
print(f"pages per filing: median {index_df.groupby('company').size().median():.0f}, "
      f"max {index_df.groupby('company').size().max()}")
print(f"\nbase model said table_found=True on {int((index_df.ocr_found == True).sum())} pages, "
      f"False on {int((index_df.ocr_found == False).sum())}, "
      f"no OCR record for {int((~index_df.has_ocr).sum())}")
index_df.head()

717 pages across 116 companies
pages per filing: median 4, max 21

base model said table_found=True on 280 pages, False on 430, no OCR record for 7


,key,company,page_file,page_no,image_path,has_ocr,ocr_found,ocr_n_tables
0,01324162/page_001.png,01324162,page_001.png,1,d:\University\Masters\Project\team_21_lloyds\D...,True,False,0
1,01324162/page_002.png,01324162,page_002.png,2,d:\University\Masters\Project\team_21_lloyds\D...,True,True,1
2,01324162/page_003.png,01324162,page_003.png,3,d:\University\Masters\Project\team_21_lloyds\D...,True,False,0
3,01324162/page_004.png,01324162,page_004.png,4,d:\University\Masters\Project\team_21_lloyds\D...,True,True,1
4,01324162/page_005.png,01324162,page_005.png,5,d:\University\Masters\Project\team_21_lloyds\D...,True,False,0


In [4]:
# --- cell 4: annotation store -------------------------------------------
# One file per company. Atomic writes: a crash mid-save can't truncate the file.

VALID_STATUS = {'pending', 'verified', 'corrected', 'skipped'}

# returns the path to the annotation file for a given company
def annot_path(company: str) -> Path:
    return ANNOT_DIR / f'{company}.json'

# loads the annotations for a given company from the corresponding JSON file
def load_annots(company: str) -> dict:
    f = annot_path(company)
    if not f.exists():
        return {}
    return json.loads(f.read_text(encoding='utf-8'))


def save_annots(company: str, data: dict) -> None:
    f = annot_path(company)
    fd, tmp = tempfile.mkstemp(dir=str(ANNOT_DIR), suffix='.tmp')
    try:
        with os.fdopen(fd, 'w', encoding='utf-8') as fh:
            json.dump(data, fh, indent=2, ensure_ascii=False)
        shutil.move(tmp, f)
    finally:
        if os.path.exists(tmp):
            os.unlink(tmp)

# retrieves the annotation for a given page key (company/page_file)
def get_annot(key: str):
    company, page = key.split('/', 1)
    return load_annots(company).get(page)

# updates the annotation for a given page key (company/page_file) with the provided payload, status, and optional note
def put_annot(key: str, payload: dict, status: str, note: str = '') -> None:
    assert status in VALID_STATUS, status
    company, page = key.split('/', 1)
    data = load_annots(company)
    data[page] = {
        'status':  status,
        'note':    note,
        'payload': payload,
        'source':  data.get(page, {}).get('source', 'model_draft'),
    }
    save_annots(company, data)

# returns the status of the annotation for a given page key (company/page_file)
def status_of(key: str) -> str:
    a = get_annot(key)
    return a['status'] if a else 'pending'

print(f'{len(list(ANNOT_DIR.glob("*.json")))} companies already have annotations')

60 companies already have annotations


In [5]:
# --- cell 5: validation --------------------------------------------------
# Catches exactly the failures in the old notebook: ragged rows, mangled
# labels from a stray find-and-replace, and table_found out of sync.

CORRUPT_RE = re.compile(r'mp_ts|tmp_table|_tmp|\bNone\b(?=\w)')

def validate(payload) -> list:
    errs = []
    if not isinstance(payload, dict):
        return ['top level is not an object']
    if 'table_found' not in payload:
        return ['missing "table_found"']
    tf = payload['table_found']
    if not isinstance(tf, bool):
        errs.append('"table_found" must be true or false')

    tables = payload.get('tables', [])
    if tf and not tables:
        errs.append('table_found is true but "tables" is empty')
    if tf is False and tables:
        errs.append('table_found is false but "tables" is non-empty')
    if not isinstance(tables, list):
        return errs + ['"tables" must be a list']

    for ti, t in enumerate(tables):
        tag = f'table[{ti}]'
        if not isinstance(t, dict):
            errs.append(f'{tag} is not an object'); continue
        name = t.get('table_name')
        if not name or not str(name).strip():
            errs.append(f'{tag} has no table_name')
        elif CORRUPT_RE.search(str(name)):
            errs.append(f'{tag} table_name looks corrupted: {name!r}')

        headers = t.get('headers')
        rows = t.get('rows')
        if not isinstance(rows, list) or not rows:
            errs.append(f'{tag} has no rows'); continue
        if headers is not None and not isinstance(headers, list):
            errs.append(f'{tag} headers must be a list or null'); continue

        if isinstance(headers, list):
            for h in headers:
                if h is not None and CORRUPT_RE.search(str(h)):
                    errs.append(f'{tag} header looks corrupted: {h!r}')
            n = len(headers)
            bad = [ri for ri, r in enumerate(rows)
                   if not isinstance(r, list) or len(r) != n]
            if bad:
                shown = ', '.join(
                    f'row {ri} has {len(rows[ri]) if isinstance(rows[ri], list) else "?"}'
                    for ri in bad[:5])
                errs.append(f'{tag} expects {n} cells per row; {shown}'
                            + (f' (+{len(bad)-5} more)' if len(bad) > 5 else ''))
        else:
            widths = {len(r) for r in rows if isinstance(r, list)}
            if len(widths) > 1:
                errs.append(f'{tag} has no headers and ragged rows: widths {sorted(widths)}')
    return errs

# self-test
assert validate({'table_found': False}) == []
assert validate({'table_found': True, 'tables': [
    {'table_name': 'x', 'headers': ['a', 'b'], 'rows': [['1', '2'], ['3']]}]})
print('validator OK')

validator OK


In [6]:
# --- cell 6: QA sweep of existing labels --------------------------------
# Run this over anything you already annotated. It will find the pages the
# old notebook silently failed to write (missing-comma TypeError aborts) and
# any labels damaged by the identifier rename.

# runs over all existing annotation files and validates their payloads
def qa_sweep(verbose_limit=25):
    problems, clean = [], 0
    for f in sorted(ANNOT_DIR.glob('*.json')):
        company = f.stem
        for page, a in load_annots(company).items():
            errs = validate(a.get('payload'))
            if errs:
                problems.append((f'{company}/{page}', a.get('status'), errs))
            else:
                clean += 1
    print(f'{clean} pages clean, {len(problems)} with problems\n')
    for key, st, errs in problems[:verbose_limit]:
        print(f'  {key}  [{st}]')
        for e in errs:
            print(f'      - {e}')
    if len(problems) > verbose_limit:
        print(f'  ... +{len(problems) - verbose_limit} more')
    return problems

qa_problems = qa_sweep()

264 pages clean, 0 with problems



In [7]:
# --- cell 7: build the work queue ---------------------------------------
# Stratified so the labelled set reflects the population you deploy on,
# not just the pages the base model already liked.

def build_queue(n_companies=60,
                positive_per_company=None,   # None = all model-positive pages
                negatives_per_company=2,     # pages the model said had no table
                only_unannotated=True,
                company_filter=None):
    df = index_df.copy()
    if company_filter is not None:
        df = df[df.company.isin(company_filter)]

    companies = sorted(df.company.unique())
    random.Random(SEED).shuffle(companies)
    companies = companies[:n_companies]

    queue = []
    for company in companies:
        sub = df[df.company == company]
        pos = sub[sub.ocr_found == True]
        neg = sub[sub.ocr_found != True]
        if positive_per_company is not None:
            pos = pos.head(positive_per_company)
        neg = neg.sample(min(negatives_per_company, len(neg)),
                         random_state=SEED) if len(neg) else neg
        picked = pd.concat([pos, neg]).sort_values('page_no')
        queue += picked.key.tolist()

    if only_unannotated:
        queue = [k for k in queue if status_of(k) == 'pending']
    return queue

QUEUE = build_queue(n_companies=60, negatives_per_company=2)
print(f'{len(QUEUE)} pages queued across '
      f'{len({k.split("/")[0] for k in QUEUE})} companies')
print('first few:', QUEUE[:5])

0 pages queued across 0 companies
first few: []


In [11]:
# --- cell 8: annotation UI ----------------------------------------------
# Image left, editable JSON right. Validate before every save.
#   Verify   = model output was already correct
#   Save     = you edited it (marked "corrected")
#   No table = shortcut for a negative
#   Skip     = ambiguous, revisit later

import ipywidgets as W
from IPython.display import display, clear_output

class Annotator:
    def __init__(self, queue, img_width=620):
        self.queue, self.i, self.img_width = queue, 0, img_width
        self._build()
        self.load()

    def _build(self):
        self.img      = W.Image(width=self.img_width)
        self.editor   = W.Textarea(layout=W.Layout(width='100%', height='560px'))
        self.editor.add_class('mono')
        self.header   = W.HTML()
        self.msg      = W.HTML()
        self.note     = W.Text(placeholder='note (optional)',
                               layout=W.Layout(width='100%'))

        mk = lambda d, s, cb, w='auto': self._btn(d, s, cb, w)
        b_prev  = mk('< Prev',   '',        lambda b: self.step(-1))
        b_next  = mk('Next >',   '',        lambda b: self.step(1))
        b_ver   = mk('Verify',   'success', lambda b: self.commit('verified'))
        b_save  = mk('Save edit','primary', lambda b: self.commit('corrected'))
        b_none  = mk('No table', 'warning', lambda b: self.mark_empty())
        b_skip  = mk('Skip',     'danger',  lambda b: self.commit('skipped', validate_first=False))
        b_fmt   = mk('Reformat', '',        lambda b: self.reformat())
        b_reset = mk('Reset',    '',        lambda b: self.load(force_draft=True))

        left  = W.VBox([self.img])
        right = W.VBox([
            W.HBox([b_ver, b_save, b_none, b_skip]),
            W.HBox([b_prev, b_next, b_fmt, b_reset]),
            self.editor, self.note, self.msg,
        ], layout=W.Layout(width='52%'))
        self.ui = W.VBox([self.header, W.HBox([left, right])])
        display(W.HTML('<style>.mono textarea{font-family:monospace;font-size:12px}</style>'))
        display(self.ui)

    @staticmethod
    def _btn(desc, style, cb, width='auto'):
        b = W.Button(description=desc, button_style=style,
                     layout=W.Layout(width=width))
        b.on_click(cb)
        return b

    # -- state -----------------------------------------------------------
    @property
    def key(self):
        return self.queue[self.i]

    def row(self):
        return index_df.set_index('key').loc[self.key]

    def load(self, force_draft=False):
        if not self.queue:
            self.header.value = '<b>Queue is empty.</b>'
            return
        r = self.row()
        self.img.value = Path(r.image_path).read_bytes()

        existing = None if force_draft else get_annot(self.key)
        if existing:
            payload, src = existing['payload'], f"saved ({existing['status']})"
            self.note.value = existing.get('note', '')
        else:
            ocr = load_ocr(r.company).get(r.page_file)
            payload = ocr if isinstance(ocr, dict) else {'table_found': False}
            src = 'model draft'
            self.note.value = ''

        self.editor.value = json.dumps(payload, indent=2, ensure_ascii=False)
        done = sum(1 for k in self.queue if status_of(k) != 'pending')
        self.header.value = (
            f"<h3 style='margin:2px'>{self.key}</h3>"
            f"<code>{self.i+1}/{len(self.queue)} &nbsp;|&nbsp; {done} done "
            f"&nbsp;|&nbsp; showing: {src}</code>")
        self.msg.value = ''

    def step(self, d):
        self.i = max(0, min(len(self.queue) - 1, self.i + d))
        self.load()

    def goto(self, i):
        self.i = max(0, min(len(self.queue) - 1, i))
        self.load()

    # -- actions ---------------------------------------------------------
    def _parse(self):
        try:
            return json.loads(self.editor.value), None
        except json.JSONDecodeError as e:
            return None, f'JSON does not parse: {e}'

    def reformat(self):
        p, err = self._parse()
        if err:
            self.say(err, bad=True); return
        self.editor.value = json.dumps(p, indent=2, ensure_ascii=False)
        self.say('reformatted')

    def mark_empty(self):
        self.editor.value = json.dumps({'table_found': False}, indent=2)
        self.commit('corrected')

    def commit(self, status, validate_first=True):
        p, err = self._parse()
        if err:
            self.say(err, bad=True); return
        if validate_first:
            errs = validate(p)
            if errs:
                self.say('NOT SAVED:<br>' + '<br>'.join('- ' + e for e in errs), bad=True)
                return
        put_annot(self.key, p, status, self.note.value)
        self.say(f'saved as <b>{status}</b>')
        if self.i < len(self.queue) - 1:
            self.step(1)

    def say(self, text, bad=False):
        colour = '#b00' if bad else '#070'
        self.msg.value = f"<div style='color:{colour}'>{text}</div>"

ann = Annotator(QUEUE)

HTML(value='<style>.mono textarea{font-family:monospace;font-size:12px}</style>')

In [10]:
# --- cell 9: progress ----------------------------------------------------
def progress():
    rows = []
    for f in sorted(ANNOT_DIR.glob('*.json')):
        c = Counter(a['status'] for a in load_annots(f.stem).values())
        rows.append({'company': f.stem, **c, 'total': sum(c     .values())})
    if not rows:
        print('nothing annotated yet'); return pd.DataFrame()
    df = pd.DataFrame(rows).fillna(0).set_index('company')
    print(df.sum(numeric_only=True).astype(int).to_string())
    print(f'\ncompanies touched: {len(df)}')
    labelled = df.get('verified', 0) + df.get('corrected', 0)
    print(f'usable pages (verified+corrected): {int(labelled.sum())}')
    return df

progress_df = progress()

verified    264
total       264

companies touched: 60
usable pages (verified+corrected): 264
